# 06 — Embeddings: Tekst om til tall

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 02, 04

**Hva du bygger:** Et system som finner de mest relevante pensjonsdokumentene for et spørsmål — uten nøkkelordmatch, bare matematikk på vektorer. Gratis med `sentence-transformers`.

---

## Hva er en embedding?

En embedding er en tekst oversatt til en liste med tall — en **vektor** — der posisjonen i rommet representerer *meningen*.

```
"AFP pensjon"        → [0.12, -0.34, 0.87, ..., 0.03]  (768 tall)
"avtalefestet pensjon" → [0.13, -0.31, 0.89, ..., 0.04]  (nesten identisk!)
"fly til Tromsø"     → [-0.72, 0.55, -0.23, ..., 0.91]  (helt annerledes)
```

**Nøkkelpoeng:** Tekster med lik *mening* havner nær hverandre i vektorrommet — selv om de bruker forskjellige ord.

In [ ]:
%pip install -q sentence-transformers numpy matplotlib

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Gratis, kjører lokalt — ingen API-nøkkel
# Første gang: laster ned ~90 MB modell automatisk
modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
# Støtter norsk + 50 andre språk

print(f"Embedding-dimensjon: {modell.get_sentence_embedding_dimension()}")

---

## Del 1: Lag embeddings og mål likhet

In [ ]:
setninger = [
    "AFP gir deg rett til tidligpensjon",
    "Avtalefestet pensjon lar deg gå av tidlig",    # Synonym for AFP
    "Alderspensjon utbetales fra 67 år",
    "Du kan ta ut pensjon fra fylte 67",             # Synonym for alderspensjon
    "Fly til Oslo med SAS koster 800 kroner",        # Helt irrelevant
]

vektorer = modell.encode(setninger)  # Shape: (5, 384)
print(f"Vektormatrise: {vektorer.shape}  ({len(setninger)} setninger × {vektorer.shape[1]} dimensjoner)")

In [ ]:
def cosinus_likhet(v1: np.ndarray, v2: np.ndarray) -> float:
    """1.0 = identisk, 0.0 = urelatert, -1.0 = motsatt."""
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

print("Likhet mellom setningspar:")
print()

par = [
    (0, 1, "AFP vs. Avtalefestet pensjon"),
    (2, 3, "Alderspensjon vs. pensjon fra 67"),
    (0, 2, "AFP vs. Alderspensjon"),
    (0, 4, "AFP vs. Flyreise"),
]

for i, j, label in par:
    likhet = cosinus_likhet(vektorer[i], vektorer[j])
    søyle  = "█" * int(likhet * 30)
    print(f"{label:<40} {søyle} {likhet:.3f}")

---

## Del 2: Semantisk søk

Finn dokumenter som svarer på et spørsmål — uten at de trenger å inneholde de samme ordene.

In [ ]:
# Simulert pensjonsdokumentbase
dokumenter = [
    "AFP (avtalefestet pensjon) gir rett til tidligpensjon fra 62 år for ansatte i offentlig sektor.",
    "Alderspensjon fra Statens pensjonskasse utbetales livsvarig fra 67 år.",
    "Uførepensjon kan innvilges ved varig nedsatt arbeidsevne på minst 20 prosent.",
    "Barnepensjon utbetales til barn under 20 år når en forsørger er død.",
    "Medlemmer kan ta opp boliglån i SPK til gunstig rente.",
    "Pensjonsopptjening skjer gjennom å betale pensjonsinnskudd via arbeidsgiver.",
]

# Embed alle dokumenter på forhånd (dyrt å gjøre hver gang)
dok_vektorer = modell.encode(dokumenter)

def søk(spørsmål: str, topp_k: int = 3) -> list:
    spørsmål_vektor = modell.encode([spørsmål])[0]
    likheter = [
        (cosinus_likhet(spørsmål_vektor, dv), dok)
        for dv, dok in zip(dok_vektorer, dokumenter)
    ]
    return sorted(likheter, reverse=True)[:topp_k]

spørsmål = "Kan jeg slutte å jobbe før jeg er 67?"
print(f"Spørsmål: {spørsmål}\n")
print("Topp treff:")
for score, dok in søk(spørsmål):
    print(f"  [{score:.3f}] {dok}")

In [ ]:
# Test med et annet spørsmål
spørsmål2 = "Hva skjer med barna mine om jeg dør?"
print(f"Spørsmål: {spørsmål2}\n")
for score, dok in søk(spørsmål2):
    print(f"  [{score:.3f}] {dok}")

---

## Del 3: Chunking-strategier

Lange dokumenter må deles opp før embedding. Størrelsen på chunks påvirker søkekvaliteten.

In [ ]:
langt_dok = """AFP (Avtalefestet pensjon) er en pensjonsordning for offentlig ansatte.
Den gir rett til å gå av med pensjon fra 62 år.
For å ha rett til AFP må du ha vært ansatt i offentlig sektor i minst tre år.
AFP utbetales livsvarig og kombineres med alderspensjon fra 67 år.
Størrelsen på AFP avhenger av lønn og opptjeningstid.
Du søker om AFP via din arbeidsgiver eller direkte til SPK.
Søknadsfristen er normalt tre måneder før ønsket uttaksdato."""

def chunk_setningsvis(tekst: str) -> list[str]:
    """Del opp på setningsgrenser — bevarer fullstendige tanker."""
    return [s.strip() for s in tekst.split(".") if s.strip()]

def chunk_fast_størrelse(tekst: str, ord_per_chunk: int = 20) -> list[str]:
    """Del i faste ordbiter — rask men kan kutte midt i setning."""
    ord = tekst.split()
    return [" ".join(ord[i:i+ord_per_chunk]) for i in range(0, len(ord), ord_per_chunk)]

chunks_setning = chunk_setningsvis(langt_dok)
chunks_fast    = chunk_fast_størrelse(langt_dok, 15)

print(f"Setningsvis ({len(chunks_setning)} chunks):")
for c in chunks_setning:
    print(f"  → {c}")

print(f"\nFast størrelse ({len(chunks_fast)} chunks):")
for c in chunks_fast:
    print(f"  → {c}")

---

## Miniprosjekt: Bygg et mini-søkesystem

In [ ]:
# Chunk dokumentet, embed alle chunks, og søk
chunks = chunk_setningsvis(langt_dok)
chunk_vektorer = modell.encode(chunks)

def søk_i_chunks(spørsmål: str, topp_k: int = 2) -> list:
    sv = modell.encode([spørsmål])[0]
    likheter = [(cosinus_likhet(sv, cv), chunk) for cv, chunk in zip(chunk_vektorer, chunks)]
    return sorted(likheter, reverse=True)[:topp_k]

for q in ["Når kan jeg søke om AFP?", "Hva er kravet for å få AFP?"]:
    print(f"Q: {q}")
    for score, chunk in søk_i_chunks(q):
        print(f"  [{score:.3f}] {chunk}")
    print()

---

## Oppsummering

| Konsept | Hva det er |
|---------|----------|
| Embedding | Tekst → tallvektor som representerer mening |
| Cosinus-likhet | Mål på vinkel mellom vektorer (0–1) |
| Semantisk søk | Finn relevante tekster uten nøkkelordmatch |
| Chunking | Del lange dokumenter før embedding |
| `sentence-transformers` | Gratis, lokal embedding-modell |

---

## Hva er neste steg?

**Neste: `07_vector_databases.ipynb`** — Nå som du kan lage embeddings, trenger du et sted å lagre og søke i dem effektivt. ChromaDB gir deg en lokal vektordatabase på minutter.